In [31]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import make_column_transformer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

DATA_PATH = os.path.join('..', 'vr_legibility_train.csv')

In [32]:
# 1. 데이터 로드 및 전처리
df = pd.read_csv(DATA_PATH)[["Yaw", "Pitch", "Size", "Label"]]

# 특성(X)과 타겟(y) 분리
X = df.drop('Label', axis=1)
y = df['Label']

# 2. 학습/테스트 데이터 분할 (Stratified Sampling)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)

# 학습/테스트 데이터 분할 (StratifiedKFold 적용)
outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

In [33]:
# 2. 파이프라인 (스케일링 + 모델) 구축
# ---------------------------------------------------------
preprocessor = make_column_transformer(
    (MinMaxScaler(), ['Yaw', 'Pitch']),
    (make_pipeline(FunctionTransformer(np.log1p), MinMaxScaler()), ['Size'])
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('lr', LogisticRegression(max_iter=5000, class_weight='balanced', random_state=42))
])

In [ ]:
#그리드 서치 하이퍼파라미터 범위
C_range = np.logspace(-8, 0, 100)
param_grid = [
    # 조합 4: 규제 없음 (Penalty None - 베이스라인 성능 확인용)
    {
        'lr__penalty': [None],
        'lr__solver': ['lbfgs', 'newton-cg', 'sag', 'saga']
    },
    # 조합 1: L2 규제 (가장 일반적) + L2를 지원하는 모든 Solver
    {
        'lr__penalty': ['l2'],
        'lr__C': C_range,
        'lr__solver': ['lbfgs', 'newton-cg', 'sag', 'saga']
    },
    # 조합 2: L1 규제 (변수 선택 효과) + L1을 지원하는 Solver
    {
        'lr__penalty': ['l1'],
        'lr__C': C_range,
        'lr__solver': ['liblinear', 'saga']
    },
    # 조합 3: ElasticNet (L1 + L2 혼합) + 전용 Solver
    {
        'lr__penalty': ['elasticnet'],
        'lr__C': C_range,
        'lr__solver': ['saga'],
        'lr__l1_ratio': np.linspace(0.1, 0.9, 20) # L1 규제 비율을 0.1부터 0.9까지 9단계로 탐색
    }
]

In [35]:
# 그리드 서치 객체 생성
grid_search = GridSearchCV(
    estimator=pipeline, 
    param_grid=param_grid, 
    cv=5, 
    scoring='f1', # 모델 평가 기준 (만약 데이터 불균형이 심하면 'f1'으로 변경)
    n_jobs=-1,          # 컴퓨터의 모든 CPU 코어를 사용하여 연산 속도 최대화
    verbose=1           # 진행 상황 출력
)

In [36]:
import warnings
warnings.filterwarnings('ignore') # 경고 메시지 무시 

f1_scores = []
acc_scores = []
auc_scores = []

for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X_train, y_train)):
    print(f"\n--- Outer Fold {fold_idx + 1} / 10 ---")
    X_train_val, X_test_val = X_train.iloc[train_idx], X_train.iloc[test_idx]
    y_train_val, y_test_val = y_train.iloc[train_idx], y_train.iloc[test_idx]

    # 내부 그리드 서치로 최적 하이퍼파라미터 탐색
    grid_search.fit(X_train, y_train)

    best_params_inner = grid_search.best_params_
    best_score_inner = grid_search.best_score_
    
    print("-" * 40)
    print(f"Best Hyper Params: {best_params_inner}")
    print(f"Best Inner CV F1:  {best_score_inner:.4f}")

    # 하이퍼파라미터로 모델 학습
    best_model_inner = pipeline.set_params(**best_params_inner)
    best_model_inner.fit(X_train_val, y_train_val)

    # 검증 데이터 평가
    y_pred = best_model_inner.predict(X_test_val)
    y_score = best_model_inner.decision_function(X_test_val)

    f1 = f1_score(y_test_val, y_pred)
    acc = accuracy_score(y_test_val, y_pred)
    roc_auc = roc_auc_score(y_test_val, y_score)

    #최고성능 하이퍼파라미터로 갱신
    if fold_idx == 0:
        best_param=best_params_inner
        best_f1=f1
    elif f1 > best_f1:
        best_param=best_params_inner
        best_f1=f1

    f1_scores.append(f1)
    acc_scores.append(acc)
    auc_scores.append(roc_auc)
    print(f"\n★ Outer fold Score:: F1 Score: {f1:.4f}, Accuracy: {acc:.4f}, ROC AUC: {roc_auc:.4f}")

print("\n"+"-" * 10 + " FINAL RESULTS " + "-" * 10)
print(f"최종 10-Fold 평균 F1 Score: {np.mean(f1_scores):.4f} (± {np.std(f1_scores):.4f})")
print(f"최종 10-Fold 평균 Accuracy: {np.mean(acc_scores):.4f} (± {np.std(acc_scores):.4f})")
print(f"최종 10-Fold 평균 ROC AUC: {np.mean(auc_scores):.4f} (± {np.std(auc_scores):.4f})")
print(f"최고 F1 하이퍼파라미터: {best_param}")


--- Outer Fold 1 / 10 ---
Fitting 5 folds for each of 1504 candidates, totalling 7520 fits
----------------------------------------
Best Hyper Params: {'lr__C': np.float64(2.9836472402833404e-05), 'lr__penalty': 'l2', 'lr__solver': 'sag'}
Best Inner CV F1:  0.7340

★ Outer fold Score:: F1 Score: 0.7478, Accuracy: 0.7302, ROC AUC: 0.7783

--- Outer Fold 2 / 10 ---
Fitting 5 folds for each of 1504 candidates, totalling 7520 fits
----------------------------------------
Best Hyper Params: {'lr__C': np.float64(2.9836472402833404e-05), 'lr__penalty': 'l2', 'lr__solver': 'sag'}
Best Inner CV F1:  0.7340

★ Outer fold Score:: F1 Score: 0.7169, Accuracy: 0.7016, ROC AUC: 0.7613

--- Outer Fold 3 / 10 ---
Fitting 5 folds for each of 1504 candidates, totalling 7520 fits
----------------------------------------
Best Hyper Params: {'lr__C': np.float64(2.9836472402833404e-05), 'lr__penalty': 'l2', 'lr__solver': 'sag'}
Best Inner CV F1:  0.7340

★ Outer fold Score:: F1 Score: 0.7297, Accuracy: 0.68